<center>

$\Huge \textbf{Universidad Nacional Autónoma de México}$  
$\Huge \textbf{Facultad de Ciencias}$  
<p align="center">
  <img src="https://www.icat.unam.mx/wp-content/uploads/2021/11/Copia-de-LogoUNAM.-Azul.-Fondo-transparente.png" alt="UNAM" width="200"/>
</p>

<hr style="height:3px; background-color:#0B6E4F; border:none;"/>


$\LARGE \textbf{Inteligencia Artificial}$  

$\Large \textit{Laboratorio 2.14}$  


\begin{array}{rl}
\textbf{Docente:} & Dra. Jessica Sarahi Méndez Rincón \\[6pt]
\textbf{Ayudante de laboratorio:} & Diego Eduardo Peña Villegas \\[6pt]
\textbf{Alumnos:} & Alan Joseph Diaz Quijada - No. Cuenta: 424044259
Chávez Martínez Marco Antonio - No.Cuenta:320328594
Lugo Díaz Ordaz Gretel Alexandra - No. Cuenta:321115128
Vega Alonso Diego Hazael- No. Cuenta: 321301183
Erick Luis Juárez - No. Cuenta: 321140153\\[6pt]
\textbf{Fecha de realización:} & 25/02/2026
\end{array}

</center>

# 1. Base de Conocimiento (Prolog)

La base de conocimiento define la ubicación ideal de los productos y las reglas para el diagnóstico y la planificación

# 2. Estados Iniciales y Finales

Para un programa de toma de decisiones, definimos los estados de la siguiente manera:Estado Inicial: El robot se encuentra en la "posición inicial" (centro del triángulo que forman los estantes). Los estantes tienen una distribución de productos que puede o no coincidir con la ideal.Estado Final:
- 1.  El cliente ha recibido el producto solicitado (ej. "Aquí está el refresco").
- 2.  Todos los productos observados durante la tarea han sido reubicados en sus estantes correctos según la base de conocimiento ("Productos Acomodados").

# 3. Integración en Python (Uso de pyswip)
Para conectar esto con Python, se utiliza la librería pyswip. El flujo consiste en que Python maneja la interfaz y el robot, mientras Prolog toma las decisiones lógicas.

In [ ]:
# 1. Instalar el motor de SWI-Prolog en el sistema
!apt-get install -y swi-prolog

# 2. Instalar la librería puente para Python
!pip install pyswip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swi-prolog is already the newest version (8.4.2+dfsg-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


Crear un archivo como base de conocimiento


% Ubicaciones ideales (Creencia inicial del sistema)

ideal(estante_bebidas, [cerveza, refresco]).

ideal(estante_comida, [sopa, cereal]).

ideal(estante_pan, [galletas]).


% Estado actual (Hechos dinámicos que Python actualizará tras la observación)

% Ejemplo de un estado con desorden (Caso 1.2 b)

ubicado_en(estante_bebidas, refresco).

ubicado_en(estante_bebidas, cerveza).

ubicado_en(estante_bebidas, sopa). % Producto desacomodado

ubicado_en(estante_comida, cereal).

ubicado_en(estante_pan, galletas).

% Regla de Diagnóstico: Identifica qué productos no están en su lugar

diagnostico(Producto, EstanteActual, EstanteCorrecto) :-

    ubicado_en(EstanteActual, Producto),

    ideal(EstanteCorrecto, ProductosIdeales),

    member(Producto, ProductosIdeales),

    EstanteActual \= EstanteCorrecto.


% Regla de Planificación: Genera las acciones necesarias

plan(entregar(P)) :- ubicado_en(_, P).

plan(acomodar(P, Destino)) :- diagnostico(P, _, Destino).

In [ ]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("supermercado.pl") # Carga la base de conocimiento

def ejecutar_asistente(producto_buscado):
    print(f"Buscando: {producto_buscado}")

    # 1. Consultar a Prolog por el plan de entrega
    plan_entrega = list(prolog.query(f"plan(entregar({producto_buscado}))"))

    # 2. Consultar si hay algo que reacomodar (Diagnóstico)
    objetos_desacomodados = list(prolog.query("plan(acomodar(P, Destino))"))

    if objetos_desacomodados:
        for obj in objetos_desacomodados:
            print(f"Acción: Mover {obj['P']} al {obj['Destino']}")

    if plan_entrega:
        print(f"Acción: Entregar {producto_buscado} al cliente.")
    else:
        print("Error: Producto no encontrado en el sistema.")

ejecutar_asistente("refresco")

Buscando: refresco
Acción: Mover sopa al estante_comida
Acción: Entregar refresco al cliente.


In [ ]:
from pyswip import Prolog

prolog = Prolog()
prolog.consult("supermercado.pl")

def manejar_error_busqueda(estante_visto, productos_reales):
    """
    Si el robot observa algo distinto a lo que creía,
    actualiza la base de conocimiento en tiempo real.
    """
    prolog.retractall(f"ubicado_en({estante_visto}, _)")

    for p in productos_reales:
        prolog.assertz(f"ubicado_en({estante_visto}, {p})")
    print(f"--- Base de datos actualizada tras observar {estante_visto} ---")

def optimizar_movimiento(acciones):
    """
    Implementa la restricción: Max 2 objetos a la vez.
    Agrupa acciones para minimizar viajes.
    """
    capacidad = 2
    carga_actual = []

    print("\n[Iniciando secuencia de movimientos optimizada]")
    for accion in acciones:
        # Si es entrega, se asume prioridad alta
        if 'entregar' in str(accion):
            print(f"EJECUTANDO: {accion}")
        else:
            carga_actual.append(accion)
            if len(carga_actual) == capacidad:
                print(f"VIAJE OPTIMIZADO: Cargando {carga_actual}")
                carga_actual = []

    if carga_actual:
        print(f"VIAJE FINAL: Cargando {carga_actual}")

def simular_reto():
    producto_cliente = "refresco"

    # 1. El robot va al estante de bebidas y lo ve VACÍO (Error de búsqueda)
    print("Robot llega a 'estante_bebidas'...")
    manejar_error_busqueda("estante_bebidas", [])

    # 2. El robot observa otros estantes y descubre la realidad
    manejar_error_busqueda("estante_comida", ["cerveza", "sopa", "galletas"])
    manejar_error_busqueda("estante_pan", ["refresco", "cereal"])

    # 3. Disparar nueva planificación con la prioridad establecida
    print("\nGenerando plan de acción...")
    resultado = list(prolog.query(f"plan_maestro({producto_cliente}, Lista)"))

    if resultado:
        todas_las_acciones = resultado[0]['Lista']
        optimizar_movimiento(todas_las_acciones)

simular_reto()

Robot llega a 'estante_bebidas'...
--- Base de datos actualizada tras observar estante_bebidas ---
--- Base de datos actualizada tras observar estante_comida ---
--- Base de datos actualizada tras observar estante_pan ---

Generando plan de acción...

[Iniciando secuencia de movimientos optimizada]
EJECUTANDO: entregar(refresco)
VIAJE OPTIMIZADO: Cargando ['acomodar(cerveza, estante_bebidas)', 'acomodar(galletas, estante_pan)']
VIAJE OPTIMIZADO: Cargando ['acomodar(refresco, estante_bebidas)', 'acomodar(cereal, estante_comida)']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Escenario del Desafío:

El cliente solicita un refresco. El robot llega al "Estante de Bebidas" y lo encuentra vacío. Al inspeccionar el "Estante de Comida", descubre que allí están la cerveza, la sopa y las galletas, mientras que el refresco y el cereal están en el "Estante de Pan".
**Reto para el programador:**
Implementar una regla en Prolog que permita al robot priorizar: ¿Debe acomodar todo primero o entregar el refresco cuanto antes para satisfacer al cliente?.

Manejar el "Error de Búsqueda": Si el robot va a un estante esperando algo y no lo encuentra, debe disparar un nuevo diagnóstico (re-planificación en tiempo real).

Restricción: El robot solo puede cargar un máximo de dos objetos a la vez. ¿Cómo optimiza el movimiento entre los 3 estantes para minimizar la distancia recorrida?

<hr/>
<footer style="text-align:center; font-size:12px; color:gray;">
© 2026 UNAM Facultado de Ciencias – Todos los derechos reservados

</footer>